In [ ]:
# In[1]:

import pandas as pd

# Load the telemetry data
metric_df = pd.read_csv('metric.csv')
log_df = pd.read_csv('log.csv')
trace_df = pd.read_csv('trace.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Convert timestamp to datetime
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Filter for possible root cause components
components = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']
metric_df = metric_df[metric_df['cmdb_id'].isin(components)]
log_df = log_df[log_df['cmdb_id'].isin(components)]
trace_df = trace_df[trace_df['cmdb_id'].isin(components)]

# Aggregate metrics by component and KPI
metric_agg = metric_df.groupby(['cmdb_id', 'kpi_name']).agg(
    count=('value', 'count'),
    min_value=('value', 'min'),
    max_value=('value', 'max'),
    mean_value=('value', 'mean'),
    p95_value=('value', lambda x: x.quantile(0.95))
).reset_index()

log_agg = log_df.groupby(['cmdb_id', 'log_name']).agg(
    count=('value', 'count'),
    min_value=('value', 'min'),
    max_value=('value', 'max'),
    mean_value=('value', 'mean'),
    p95_value=('value', lambda x: x.quantile(0.95))
).reset_index()

trace_agg = trace_df.groupby(['cmdb_id', 'trace_name']).agg(
    count=('value', 'count'),
    min_value=('value', 'min'),
    max_value=('value', 'max'),
    mean_value=('value', 'mean'),
    p95_value=('value', lambda x: x.quantile(0.95))
).reset_index()

# Combine all aggregated data into a single DataFrame
combined_agg = pd.concat([metric_agg, log_agg.rename(columns={'log_name': 'kpi_name'}), trace_agg.rename(columns={'trace_name': 'kpi_name'})], ignore_index=True)

# Display the combined aggregated results
combined_agg_summary = combined_agg.groupby(['cmdb_id', 'kpi_name']).agg(
    total_count=('count', 'sum'),
    earliest_timestamp=('min_value', 'min'),
    max_value=('max_value', 'max'),
    min_value=('min_value', 'min'),
    mean_value=('mean_value', 'mean'),
    p95_value=('p95_value', 'mean')
).reset_index()

combined_agg_summary = combined_agg_summary.head(20)
combined_agg_summary

```
Out[1]:
```
The aggregation of KPIs for the possible root cause components has been completed. The results include various metrics such as total counts, earliest timestamps, maximum and minimum values, mean values, and the 95th percentile values for each component and KPI. The data is summarized for the top 20 entries, providing insights into the performance and potential issues across the components of the Train Ticket system.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                  kpi_name  total_count  earliest_timestamp     max_value     min_value    mean_value     p95_value
0   ts-auth-service                                       cpu           25        1.333997e+01  3.979049e+01  1.333997e+01  2.580557e+01  3.973440e+01
1   ts-auth-service                                    diskio           25        3.631332e+04  1.636124e+05  3.631332e+04  5.837022e+04  1.055891e+05
2   ts-auth-service                                latency-50           25        2.087717e-01  8.917512e-01  2.087717e-01  5.063298e-01  8.772784e-01
3   ts-auth-service                                latency-90           25        4.489010e-01  2.267387e+00  4.489010e-01  1.185696e+00  2.046220e+00
4   ts-auth-service                           log.error_count           25        0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
5   ts-auth-service                             log.row_count           25        4.800000e+01  3.520000e+02  4.800000e+01  3.119200e+02  3.436000e+02
6   ts-auth-service                                       mem           25        2.576880e+08  2.638291e+08  2.576880e+08  2.602235e+08  2.634673e+08
7   ts-auth-service                                    socket           25        1.128333e+01  1.938333e+01  1.128333e+01  1.426414e+01  1.712179e+01
8   ts-auth-service             trace.from_root.duration_mean           25        2.324917e-01  1.040964e+00  2.324917e-01  5.678032e-01  9.632253e-01
9   ts-auth-service              trace.from_root.duration_p95           25        3.591124e-01  2.362664e+00  3.591124e-01  1.114787e+00  1.852401e+00
10  ts-auth-service                trace.from_root.error_rate           25        0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
11  ts-auth-service                 trace.from_root.row_count           25        1.200000e+01  8.900000e+01  1.200000e+01  7.804000e+01  8.580000e+01
12  ts-auth-service  trace.from_ts-auth-service.duration_mean           25        1.386396e-03  1.334527e-02  1.386396e-03  5.107404e-03  1.123334e-02
13  ts-auth-service   trace.from_ts-auth-service.duration_p95           25        1.671350e-03  9.588600e-02  1.671350e-03  3.359705e-02  8.672657e-02
14  ts-auth-service     trace.from_ts-auth-service.error_rate           25        0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
15  ts-auth-service      trace.from_ts-auth-service.row_count           25        4.800000e+01  3.520000e+02  4.800000e+01  3.119200e+02  3.436000e+02
16  ts-auth-service    trace.to_ts-auth-service.duration_mean           25        1.386396e-03  1.334527e-02  1.386396e-03  5.107404e-03  1.123334e-02
17  ts-auth-service     trace.to_ts-auth-service.duration_p95           25        1.671350e-03  9.588600e-02  1.671350e-03  3.359705e-02  8.672657e-02
18  ts-auth-service       trace.to_ts-auth-service.error_rate           25        0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
19  ts-auth-service        trace.to_ts-auth-service.row_count           25        4.800000e+01  3.520000e+02  4.800000e+01  3.119200e+02  3.436000e+02```
```

In [ ]:
# In[2]:

# Ensure the 'earliest_timestamp' column is in datetime format
combined_agg_summary['earliest_timestamp'] = pd.to_datetime(combined_agg_summary['earliest_timestamp'], unit='s', utc=True)

# Filter the aggregated KPI data within the incident window
filtered_agg = combined_agg_summary[
    (combined_agg_summary['earliest_timestamp'] >= incident_start) & 
    (combined_agg_summary['earliest_timestamp'] <= incident_end)
]

# Display the filtered results
filtered_agg_summary = filtered_agg.head(20)
filtered_agg_summary

```
Out[2]:
```
The filtering of the aggregated KPI data for the specified incident window resulted in an empty dataset, indicating that there were no entries recorded within the time frame of 2024-01-23 15:11:51 to 2024-01-23 15:41:51 UTC.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, kpi_name, total_count, earliest_timestamp, max_value, min_value, mean_value, p95_value]
Index: []```
```

In [ ]:
# In[3]:

# Load the log data
log_df = pd.read_csv('log.csv')

# Convert timestamp to datetime
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Define the incident window
incident_start = pd.to_datetime('2024-01-23 15:11:51', utc=True)
incident_end = pd.to_datetime('2024-01-23 15:41:51', utc=True)

# Filter log data within the incident window
filtered_log = log_df[
    (log_df['timestamp'] >= incident_start) & 
    (log_df['timestamp'] <= incident_end)
]

# Aggregate log data for counts and earliest timestamps
log_agg = filtered_log.groupby('log_name').agg(
    total_count=('value', 'count'),
    earliest_timestamp=('timestamp', 'min'),
    max_value=('value', 'max'),
    min_value=('value', 'min'),
    mean_value=('value', 'mean')
).reset_index()

# Display the aggregated log results
log_agg_summary = log_agg.head(20)
log_agg_summary

```
Out[3]:
```
The analysis of the log data within the incident window revealed two log entries: 

1. **Error Count**: There were 1,102 entries, with the earliest timestamp at 2024-01-23 15:14:00 UTC. The maximum and minimum values were both 0, indicating no errors were recorded during this period.

2. **Row Count**: There were also 1,102 entries, with the earliest timestamp at the same time. The maximum row count was 2,737, with a minimum of 1 and an average of approximately 246.75 rows.

This suggests that while no errors occurred, there was a significant volume of log entries related to row counts during the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

log_name  total_count        earliest_timestamp  max_value  min_value  mean_value
0  log.error_count         1102 2024-01-23 15:14:00+00:00          0          0    0.000000
1    log.row_count         1102 2024-01-23 15:14:00+00:00       2737          1  246.750454```
```

In [ ]:
# In[4]:

# Load the trace data
trace_df = pd.read_csv('trace.csv')

# Convert timestamp to datetime
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Filter trace data within the incident window
filtered_trace = trace_df[
    (trace_df['timestamp'] >= incident_start) & 
    (trace_df['timestamp'] <= incident_end)
]

# Aggregate trace data for counts and earliest timestamps
trace_agg = filtered_trace.groupby('trace_name').agg(
    total_count=('value', 'count'),
    earliest_timestamp=('timestamp', 'min'),
    max_value=('value', 'max'),
    min_value=('value', 'min'),
    mean_value=('value', 'mean')
).reset_index()

# Identify anomalies in trace durations (e.g., using a threshold for mean value)
# Assuming a threshold for anomalies is defined as mean_value > 1.0 for demonstration
anomaly_threshold = 1.0
trace_agg['anomaly'] = trace_agg['mean_value'] > anomaly_threshold

# Display the aggregated trace results
trace_agg_summary = trace_agg.head(20)
trace_agg_summary

```
Out[4]:
```
The analysis of the trace data within the incident window revealed the following:

1. **Total Counts**: Various trace entries were recorded, with the highest count being 346 for the `trace.from_root` metrics.

2. **Earliest Timestamps**: The earliest timestamp for all entries was at 2024-01-23 15:14:00 UTC.

3. **Anomalies in Trace Durations**: 
   - The `trace.from_root.row_count`, `trace.from_ts-admin-basic-info-service.row_count`, `trace.from_ts-admin-travel-service.row_count`, and `trace.from_ts-assurance-service.row_count` metrics showed anomalies, indicating that their mean values exceeded the defined threshold.
   - The mean values for these entries were significantly higher than expected, suggesting potential issues in those areas.

Overall, while most trace metrics did not indicate anomalies, specific row count metrics raised concerns that may require further investigation.

The original code execution output of IPython Kernel is also provided below for reference:

trace_name  total_count        earliest_timestamp   max_value  min_value  mean_value  anomaly
0                       trace.from_root.duration_mean          346 2024-01-23 15:14:00+00:00    1.040964   0.003376    0.230460    False
1                        trace.from_root.duration_p95          346 2024-01-23 15:14:00+00:00    2.362664   0.003599    0.304657    False
2                          trace.from_root.error_rate          346 2024-01-23 15:14:00+00:00    0.000000   0.000000    0.000000    False
3                           trace.from_root.row_count          346 2024-01-23 15:14:00+00:00   89.000000   1.000000   18.713873     True
4   trace.from_ts-admin-basic-info-service.duratio...           75 2024-01-23 15:14:00+00:00    0.006414   0.002829    0.004083    False
5   trace.from_ts-admin-basic-info-service.duratio...           75 2024-01-23 15:14:00+00:00    0.009196   0.002955    0.004926    False
6   trace.from_ts-admin-basic-info-service.error_rate           75 2024-01-23 15:14:00+00:00    0.000000   0.000000    0.000000    False
7    trace.from_ts-admin-basic-info-service.row_count           75 2024-01-23 15:14:00+00:00   21.000000   1.000000   11.706667     True
8    trace.from_ts-admin-travel-service.duration_mean           75 2024-01-23 15:14:00+00:00    0.077794   0.051675    0.057364    False
9     trace.from_ts-admin-travel-service.duration_p95           75 2024-01-23 15:14:00+00:00    0.144803   0.053021    0.066690    False
10      trace.from_ts-admin-travel-service.error_rate           75 2024-01-23 15:14:00+00:00    0.000000   0.000000    0.000000    False
11       trace.from_ts-admin-travel-service.row_count           75 2024-01-23 15:14:00+00:00   20.000000   2.000000   11.733333     True
12      trace.from_ts-assurance-service.duration_mean           25 2024-01-23 15:14:00+00:00    0.004986   0.001677    0.002100    False
13       trace.from_ts-assurance-service.duration_p95           25 2024-01-23 15:14:00+00:00    0.003704   0.002420    0.003086    False
14         trace.from_ts-assurance-service.error_rate           25 2024-01-23 15:14:00+00:00    0.000000   0.000000    0.000000    False
15          trace.from_ts-assurance-service.row_count           25 2024-01-23 15:14:00+00:00   28.000000   4.000000   10.880000     True
16           trace.from_ts-auth-service.duration_mean           25 2024-01-23 15:14:00+00:00    0.013345   0.001386    0.005107    False
17            trace.from_ts-auth-service.duration_p95           25 2024-01-23 15:14:00+00:00    0.095886   0.001671    0.033597    False
18              trace.from_ts-auth-service.error_rate           25 2024-01-23 15:14:00+00:00    0.000000   0.000000    0.000000    False
19               trace.from_ts-auth-service.row_count           25 2024-01-23 15:14:00+00:00  352.000000  48.000000  311.920000     True```
```

In [ ]:
# In[5]:

# Load the error log data
error_logs_df = pd.read_csv('error_logs.csv')

# Convert timestamp to datetime
error_logs_df['timestamp'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True)

# Filter error log data within the incident window
filtered_error_logs = error_logs_df[
    (error_logs_df['timestamp'] >= incident_start) & 
    (error_logs_df['timestamp'] <= incident_end)
]

# Aggregate error log data for counts and severity levels
error_agg = filtered_error_logs.groupby('message').agg(
    total_count=('timestamp', 'count'),
    earliest_timestamp=('timestamp', 'min')
).reset_index()

# Display the aggregated error log results
error_agg_summary = error_agg.head(20)
error_agg_summary

```
Out[5]:
```
The analysis of the error log data within the specified incident window resulted in an empty dataset, indicating that no error messages were recorded during the time frame of 2024-01-23 15:11:51 to 2024-01-23 15:41:51 UTC.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [message, total_count, earliest_timestamp]
Index: []```
```

In [ ]:
# In[6]:

# Re-evaluate the metrics data with loosened thresholds (P90)
# Calculate global P90 thresholds for each 'component-KPI' time series
metric_p90 = metric_df.groupby(['cmdb_id', 'kpi_name']).agg(
    p90_value=('value', lambda x: x.quantile(0.90))
).reset_index()

# Merge with the combined aggregated summary to identify anomalies
combined_agg_with_p90 = combined_agg_summary.merge(metric_p90, on=['cmdb_id', 'kpi_name'], suffixes=('', '_p90'))

# Identify anomalies based on the new P90 threshold
combined_agg_with_p90['anomaly'] = combined_agg_with_p90['mean_value'] >= combined_agg_with_p90['p90_value']

# Filter for anomalies
anomalies_p90 = combined_agg_with_p90[combined_agg_with_p90['anomaly']]

# Display the summary of anomalies
anomalies_p90_summary = anomalies_p90[['cmdb_id', 'kpi_name', 'total_count', 'earliest_timestamp', 'max_value', 'min_value', 'mean_value', 'p90_value']].head(20)
anomalies_p90_summary

```
Out[6]:
```
The re-evaluation of the metrics data with the loosened thresholds (from P95 to P90) did not reveal any anomalies. The analysis resulted in an empty dataset, indicating that no performance issues were detected during the incident window based on the new criteria.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, kpi_name, total_count, earliest_timestamp, max_value, min_value, mean_value, p90_value]
Index: []```
```